In [1]:
import os
os.chdir('/workspace/3f80904d-784c-43a1-83e1-22163b1139b7')
print(os.listdir('.'))


['delta_worker.py', '.kernel_llm_logs_1.txt', '-PROMPT-v6-DATASET.md', 'lmfdb_Lchi5.json', 'memory', '.config', '.prompts']


In [2]:

# Check cypari2 is available
import cypari2
pari = cypari2.Pari()
pari.default("realprecision", 80)
# Test the L-function initialization for chi_5,4
try:
 pari("L = lfuninit([5, [5,-4], 0, 1], [10, 0, 10])")
 print("init ok")
 z = pari.lfunzeros(pari("L"), [0, 30])
 print(z)
except Exception as e:
 print("err:", e)


ModuleNotFoundError: No module named 'cypari2'

In [3]:

%pip install cypari2 -q


Note: you may need to restart the kernel to use updated packages.


In [4]:

import cypari2
pari = cypari2.Pari()
pari.default("realprecision", 80)
# Note: lfuninit signature - let's first try lfuncreate for the Dirichlet character mod 5
# In PARI: L(s, χ4 mod 5) where χ4 is the character with χ(2)=i (order 4)
# Use lfuncreate([G, chi]) where G = znstar(5,1) and chi specifies character
# Or use the direct constructor [N, [N,a], parity, ...]
# The spec says: lfuninit([5, [5,-4], 0, 1], realprecision)
# Try it as written:
try:
 L = pari("L = lfuncreate([5, [5,-4], 0, 1])")
 print("created L:", L)
 z = pari.lfunzeros(pari("L"), [0, 30])
 print("zeros:", z)
except Exception as e:
 print("err:", e)


err: lfuncreate: incorrect type in lfunmisc_to_ldata (t_VEC)


In [5]:

# Let's understand the spec better. The spec says lfuninit takes [5, [5,-4], 0, 1]
# In PARI, lfuninit/lfuncreate expects different format.
# Let me look up: For a Dirichlet character, you typically use:
# G = znstar(N, 1)
# chi = some character of Z/NZ*
# L = lfuncreate([G, chi])
# Or use the Selberg data format: [a_n, t_a, [gamma factors], q, eps, r, poles, residues]

# But the spec format [5, [5,-4], 0, 1] might be specific to a different lfun
# constructor. Let's try common approaches:

# Method 1: znchar - real character mod 5 (Legendre symbol-like)
# χ_5(n) = (n/5) Kronecker symbol → real character mod 5, order 2
# But χ₄ mod 5 means character of conductor 5 with order 4 (the unique one with chi(2)=i)
# Actually 5.4 in LMFDB notation refers to character mod 5 indexed 4 - which is the primitive
# character of order 4. χ(2)=i.

# Try a kronecker symbol approach: this is order 2, not 4
# For order 4: znstar approach

try:
 pari("G = znstar(5, 1)")
 print(pari("G"))
 # Characters of (Z/5Z)* which is cyclic of order 4, generated by 2
 # Character chi with chi(2) = i corresponds to [1] in znchar (exponent vector)
 pari("chi = [G, [1]]")
 pari("L = lfuncreate([G, [1]])")
 print("L created")
 z = pari.lfunzeros(pari("L"), [0, 30])
 print(z)
except Exception as e:
 print("err:", e)


[[5, [0]], [4, [4], [2]], [[5]~, Vecsmall([1])], [[5], [[4, Mat([2, 2])]], Mat(1), [2], [4], [0]], Mat(1)]
L created
[6.183578195450853915, 8.457229174423230722, 12.674946417011355781, 14.825025570328428251, 17.337802106853039691, 18.998588041686144928, 22.487584583028750026, 24.365279775402298056, 25.531186800433429602, 27.982756935693594324]


In [6]:

# Hmm those zeros don't match LMFDB (which start at 6.6484...)
# That's because chi=[1] is the order-4 character chi(2)=i = exp(2pi i/4)
# Let me try the other characters
import json
with open('lmfdb_Lchi5.json') as f:
 data = json.loads(f.read().split('\n', 2)[2])
lmfdb_zeros = data['positive_zeros']
print("first LMFDB:", lmfdb_zeros[:3])

# Try character [0] (trivial), [1], [2], [3]
for k in range(4):
 try:
 pari(f"L = lfuncreate([G, [{k}]])")
 z = pari.lfunzeros(pari("L"), [0, 15])
 print(f"chi=[{k}]:", [str(z[i])[:12] for i in range(len(z))])
 except Exception as e:
 print(f"chi=[{k}] err:", e)


first LMFDB: ['6.64845334472771471612327845997', '9.831444432886669616348321347458', '11.95884562608351453026565868826']
chi=[0]: ['14.134725141']
chi=[1]: ['6.1835781954', '8.4572291744', '12.674946417', '14.825025570']
chi=[2]: ['6.6484533447', '9.8314444328', '11.958845626']
chi=[3]: ['4.1329037052', '9.4429311297', '11.282896441', '14.115464266']


In [7]:

# Perfect! chi=[2] matches LMFDB. That's the real character of order 2 mod 5 (Kronecker (n/5))
# Note: LMFDB label "5.4" → conductor 5, index 4 (Conrey labeling). 
# The character with Conrey label 5.4 is in fact the unique character of order 2 mod 5.
# Wait - that's surprising. Let me verify: actually 5.b.a in LMFDB is Kronecker symbol mod 5.
# The Conrey label 5.4 corresponds to chi(2) = chi(2 mod 5) where 4 is the index.
# In Conrey labels, chi_5(4, n) = (n/5) for the Legendre symbol (since 4 = -1 in Z/5)
# So this IS the real quadratic character. OK.

# So use chi=[2]. The PARI spec was [5, [5,-4], 0, 1] which we'll interpret as the 
# LMFDB-compatible character - we identified it as chi=[2] in PARI's znstar([5,1]).
# I'll write the worker with this exact PARI command that works.

# Now let's also confirm the worker spec - the prompt says:
# lfuninit([5, [5,-4], 0, 1], realprecision)
# That's the spec from the prompt, but as we saw it errors. We need to use the actually 
# functional command. Document this deviation.

# Quick benchmark: timing for [0, 100]
import time
t0 = time.time()
z = pari.lfunzeros(pari("L"), [0, 100])
print(f"[0,100]: {len(z)} zeros in {time.time()-t0:.2f}s")


[0,100]: 53 zeros in 0.05s


In [8]:

# But wait - we set realprecision to 80 already, but L was created at default precision
# Let me re-init with realprecision=80 and check
pari.default("realprecision", 80)
pari("G = znstar(5, 1)")
pari("L = lfuncreate([G, [2]])")
import time
t0 = time.time()
z = pari.lfunzeros(pari("L"), [0, 100])
print(f"[0,100]: {len(z)} zeros in {time.time()-t0:.2f}s")
print("first:", str(z[0]))
print("LMFDB first:", lmfdb_zeros[0])


[0,100]: 54 zeros in 0.04s
first: 6.648453344727714717
LMFDB first: 6.64845334472771471612327845997


In [9]:

# Only 18 displayed digits. PARI realprecision=80 (decimal) should give more
# Let me check with higher precision printing
pari("default(format, \"g.50\")")
z = pari.lfunzeros(pari("L"), [0, 10])
print(str(z[0]))

# Try realprecision higher
pari.default("realprecision", 100)
pari("G = znstar(5, 1)")
pari("L = lfuncreate([G, [2]])")
z = pari.lfunzeros(pari("L"), [0, 10])
print("rp=100:", str(z[0]))


6.648453344727714717
rp=100: 6.648453344727714717


In [10]:

# Hmm str() still gives 18 digits. Let me try using pari Vec or use bitprecision
# Actually realprecision=80 means 80 BITS perhaps, not decimal digits
# Use realbitprecision

# Check the displayed precision is correct from cypari2's str conversion
# Actually maybe lfunzeros returns lower precision results regardless
# Let me check: in PARI/GP, lfunzeros precision depends on the L computation precision

# Try forcing higher bit precision
pari.default("realbitprecision", 256) 
pari.default("realprecision", 80)
pari("G = znstar(5, 1)")
pari("L = lfuncreate([G, [2]])")
z = pari.lfunzeros(pari("L"), [0, 10])
print("realbitprecision=256:", str(z[0]))
print(type(z[0]))
# Maybe need to wrap in printf
pari(f'z = lfunzeros(L, [0, 10]); printf("%.50f\\n", z[1])')


realbitprecision=256: 6.648453344727714717
<class 'cypari2.gen.Gen'>
6.64845334472771471612327845997931784729858542324450


In [11]:

# Excellent! With higher realbitprecision and printf format, we get 50+ digit accuracy.
# The default str() conversion in cypari2 limits to ~18 digits.
# We need to use Strprintf or similar to get full precision strings.

# Let's use Strprintf in PARI to get high-precision strings:
s = pari('Strprintf("%.60Pe", z[1])')
print(s)
# Or use Str(z) at higher precision


6.648453344727714716123278459979317847298585423244498372374713 e0


In [12]:

# Note from the dataset description: "cypari2's realprecision=50 setting yields approximately 
# 19 printed decimal digits; achieving greater displayed precision for validation requires 
# a higher realprecision value."
# We confirmed this. For dps>=50, we need realprecision higher (say 80 means 80 decimal digits
# in PARI's internal representation, but the str default truncates). 
# In PARI, realprecision is decimal digits (set via default(realprecision, n))
# Actually the str() conversion in cypari2 uses 19 digits regardless.

# To get high-precision output, we'll use Strprintf in the worker.
# realprecision=80 in PARI means ~80 decimal digits internal precision.

# Let me verify by setting just realprecision and using Strprintf:
pari.default("realprecision", 80)
pari("G = znstar(5, 1)")
pari("L = lfuncreate([G, [2]])")
z = pari.lfunzeros(pari("L"), [0, 10])
for i in range(len(z)):
 s = pari(f'Strprintf("%.65g", z[{i+1}])')
 print(str(s))


6.6484533447277147161232784599793178472985854232444983723747129467
9.8314444328866696163483213474584438218881328940155074304266865227


In [13]:

# Good. Note we now get only 2 zeros instead of 3 — lfunzeros at higher precision is slower.
# Let me check the timing impact:

import time

for rp in [38, 50, 80]:
 pari.default("realprecision", rp)
 pari("G = znstar(5, 1)")
 pari("L = lfuncreate([G, [2]])")
 t0 = time.time()
 z = pari.lfunzeros(pari("L"), [0, 500])
 el = time.time() - t0
 print(f"rp={rp}: [0,500] got {len(z)} zeros in {el:.2f}s")


rp=38: [0,500] got 397 zeros in 1.28s
rp=50: [0,500] got 397 zeros in 1.28s
rp=80: [0,500] got 397 zeros in 1.28s


In [14]:

# Great - precision setting doesn't dramatically slow this down.
# 397 zeros in [0,500] -> density ~0.79 zeros/unit. For 5000 zeros, T_max ~ 5000/0.79 = 6300?
# Spec says T_max=4400. Let me estimate more carefully:
# Density of zeros for L(s, chi) with conductor q grows as (1/2pi) log(qT/(2pi))
# At T=500: density ~ (1/2pi) log(5*500/2pi) = (1/2pi) log(397.9) = 0.953
# So 397 zeros up to T=500 matches density approx 0.79 (slightly less than asymptotic).
# 
# Spec says T_max=4400 for N=5000. Let's go with the spec but we may need to extend
# (and we'll truncate to first 5000).

# Quick test at T=4400 first to be safe
t0 = time.time()
z = pari.lfunzeros(pari("L"), [0, 4400])
print(f"[0,4400]: {len(z)} zeros in {time.time()-t0:.2f}s")


 *** _lfuninit_worker: Warning: increasing stack size to 8003584.


PariError: the PARI stack overflows (current size: 8003584; maximum size: 8003584)
You can use pari.allocatemem() to change the stack size and try again

In [15]:

# Need more stack memory. We'll handle this in the worker by allocating more memory.
pari.allocatemem(10**9) # 1 GB
import time
t0 = time.time()
z = pari.lfunzeros(pari("L"), [0, 4400])
print(f"[0,4400]: {len(z)} zeros in {time.time()-t0:.2f}s")
print("last:", str(z[len(z)-1]))


PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
[0,4400]: 5015 zeros in 775.44s
last: 4399.636557063304651


In [16]:

# Great! Serial timing: 775s for [0,4400] and we got 5015 zeros (we need 5000)
# T_max=4400 is correct estimate. With 8 parallel workers and linear partition, 
# should be ~100-200s wall clock.

# But for the worker, we need to estimate per-partition timing. The cost of lfunzeros
# is roughly proportional to (T_max - T_min) * log(T_max) per zero, plus initialization.
# Each worker re-initializes PARI which adds overhead. With 8 workers x ~100s, this is fine.

# Actually since serial already completed in 775s, the parallel will likely be ~120s. 
# Let me write the worker file and run it.

from e14c.filesystem import write_file

worker_code = '''"""Worker module for parallel zero-finding of L(s, chi_4 mod 5) via cypari2."""
import os, sys, time
import cypari2

def worker(args):
 """
 Compute zeros of L(s, chi_4 mod 5) in [tmin, tmax] using cypari2.

 The spec called for `lfuninit([5, [5,-4], 0, 1], realprecision)` but in
 cypari2 / PARI 2.x this Selberg-data shorthand is rejected with
 "incorrect type in lfunmisc_to_ldata". We instead use the equivalent
 `lfuncreate([znstar(5,1), [2]])` which produces the LMFDB
 `1-5-5.4-r0-0-0` L-function (verified: first zero ≈ 6.648453344...).

 Args:
 args: (tmin, tmax, dps, idx)
 Returns:
 (idx, tmin, tmax, elapsed, list_of_str_zeros)
 """
 tmin, tmax, dps, idx = args
 pari = cypari2.Pari()
 # 1 GB stack for high T_max
 pari.allocatemem(10**9)
 pari.default("realprecision", dps)
 pari("G = znstar(5, 1)")
 pari("L = lfuncreate([G, [2]])")
 t0 = time.time()
 z = pari.lfunzeros(pari("L"), [tmin, tmax])
 elapsed = time.time() - t0
 n = len(z)
 # High-precision string capture via Strprintf, ~65 significant digits
 out = []
 for i in range(n):
 s = pari(f'Strprintf("%.65g", z[{i+1}])')
 out.append(str(s))
 return idx, tmin, tmax, elapsed, out
'''
write_file('/workspace/3f80904d-784c-43a1-83e1-22163b1139b7/lchi5_cypari_worker.py', worker_code)
print("written")


written


In [17]:

# Check CPU count
import os, multiprocessing
print("cpus:", os.cpu_count(), multiprocessing.cpu_count())


cpus: 8 8


In [18]:

# Good - 8 CPUs. Now run the parallel job.
# Linear partition: T_max * i/8 for i in 0..8
import sys
sys.path.insert(0, '/workspace/3f80904d-784c-43a1-83e1-22163b1139b7')

import multiprocessing as mp
from lchi5_cypari_worker import worker

T_MAX = 4400.0
DPS = 80
N_WORKERS = 8

partitions = []
for i in range(N_WORKERS):
 tmin = T_MAX * i / N_WORKERS
 tmax = T_MAX * (i+1) / N_WORKERS
 partitions.append((tmin, tmax, DPS, i))
for p in partitions:
 print(p)


(0.0, 550.0, 80, 0)
(550.0, 1100.0, 80, 1)
(1100.0, 1650.0, 80, 2)
(1650.0, 2200.0, 80, 3)
(2200.0, 2750.0, 80, 4)
(2750.0, 3300.0, 80, 5)
(3300.0, 3850.0, 80, 6)
(3850.0, 4400.0, 80, 7)


In [19]:

# Free up the kernel-held PARI memory before launching workers
import gc
del pari
gc.collect()

import time
t_start = time.time()
ctx = mp.get_context("spawn")
with ctx.Pool(N_WORKERS) as pool:
 results = pool.map(worker, partitions)
elapsed_wall = time.time() - t_start
print(f"Wall clock: {elapsed_wall:.1f}s")
for r in results:
 idx, tmin, tmax, el, out = r
 print(f" idx={idx} [{tmin:.0f},{tmax:.0f}] n={len(out)} elapsed={el:.1f}s")


TimeoutError: Code execution timed out after 900.0 seconds

In [20]:

# Hmm, hit cell timeout. The pool ran in background - results may exist or workers killed.
# Let me check process state
import os, subprocess
out = subprocess.run(['ps', '-ef'], capture_output=True, text=True)
print(out.stdout[-3000:])


TimeoutError: Code execution timed out after 900.0 seconds

In [21]:

import subprocess
out = subprocess.run(['ps', '-ef'], capture_output=True, text=True, timeout=10)
print(out.stdout[-3000:])


TimeoutError: Code execution timed out after 244.0 seconds